# KataGo Master 免費雲端 GPU 啟動器（Kaggle）

本 Notebook 可在 Kaggle 雲端工作階段中編譯並啟動真正的 KataGo 分析引擎，再透過 Cloudflare Quick Tunnel 提供臨時 HTTPS 網址。**不需要 Windows 桌機，也不需要購買網域或 Cloudflare 帳號設定。**

## 用手機操作前先知道
- 建議使用 Android 手機瀏覽器開啟 Kaggle 網頁；若版面不完整，可在瀏覽器選單開啟「桌面版網站」。這只是手機顯示網頁，不代表需要桌上型電腦。
- 先登入 Kaggle 帳號，建立或匯入這份 Notebook。GitHub 原始檔： https://raw.githubusercontent.com/song20090205-max/KataDroid/feature/katago-cloud-engine-2026-11-21/cloud/KataGo_Cloud_Kaggle_GPU.ipynb
- 在 Notebook 的 Settings / Session options 中，將 Accelerator 設為可用的 GPU（例如 GPU T4 x2），並開啟 Internet。GPU 若排隊或不可用，稍後再試；不要改用付費方案。
- 依序執行第 1 個程式碼儲存格（Setup）及第 2 個程式碼儲存格（Launch）。第一次會下載程式與模型並編譯，可能需要一段時間。
- Launch 儲存格會要求輸入至少 32 字元的 API Token；輸入時畫面不會顯示文字。請自行保管，絕不可貼到聊天或公開頁面。
- 成功後會顯示 Cloud URL。將網址與剛才的 API Token 填入 KataGo Master 的 Cloud 設定。

## 重要限制
- 免費 GPU 不是 24 小時常駐服務。Kaggle 有 GPU 配額、排隊與工作階段時間限制；工作階段停止或中斷時，API 和臨時網址都會失效。
- 每次重新啟動通常會產生新網址；需要重新複製到 App。這是免費測試／按需分析方案，不是永久伺服器。
- 不要公開分享正在運作的 Notebook、輸出內容、Cloud URL 或 API Token。停止工作階段即可關閉服務。
- 如果 GPU 沒有成功啟用，程式會退回 CPU 編譯；這不代表雲端 GPU 已啟動。以輸出中的 Backend 狀態為準。


In [ ]:
import os, pathlib, subprocess, textwrap, sys, json, time, shutil

REPO = 'https://github.com/song20090205-max/KataDroid.git'
BRANCH = 'feature/katago-cloud-engine-2026-11-21'
WORK = pathlib.Path('/kaggle/working/katago-cloud')

if not pathlib.Path('/kaggle/working').exists():
    raise RuntimeError('This notebook must run on Kaggle Notebooks.')

def run(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(cmd, cwd=cwd, check=True)

# Install Eigen development files required by KataGo's CMake configuration.
# Kaggle may have Eigen headers but omit the Eigen3 CMake package metadata.
run(['bash', '-lc', 'apt-get update -qq && apt-get install -y -qq libeigen3-dev'])

# Kaggle images usually include the compiler toolchain. Avoid reinstalling the compiler toolchain.
required = ['g++', 'make', 'cmake', 'git', 'curl', 'gzip']
missing = [tool for tool in required if shutil.which(tool) is None]
if missing:
    raise RuntimeError('Kaggle image is missing required tools: ' + ', '.join(missing) + '. Enable Internet in Notebook Settings, then restart the session; no manual code changes are needed.')
print('Build tools available:', ', '.join(required))
if WORK.exists():
    run(['rm', '-rf', str(WORK)])
run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(WORK)])

gpu_build = subprocess.run(['bash','-lc','command -v nvcc >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1'], capture_output=True).returncode == 0
katago_src = pathlib.Path('/kaggle/working/KataGo')
run(['git', 'clone', '--depth', '1', '--branch', 'v1.18.2', 'https://github.com/lightvector/KataGo.git', str(katago_src)])
cpp = katago_src / 'cpp'
backend = 'CUDA' if gpu_build else 'EIGEN'
print('Selected KataGo backend:', backend)
run(['cmake', '.', f'-DUSE_BACKEND={backend}', '-DCMAKE_BUILD_TYPE=Release', '-DNO_GIT_REVISION=1'], cwd=cpp)
run(['make', '-j2'], cwd=cpp)

cloud = WORK / 'cloud'
models = cloud / 'models'
models.mkdir(parents=True, exist_ok=True)
run(['bash', 'download_model.sh'], cwd=cloud)
model = models / 'model.bin.gz'
run(['gzip', '-t', str(model)])
config = katago_src / 'cpp' / 'configs' / 'analysis_example.cfg'
if not config.exists():
    raise FileNotFoundError(config)

print('\nSetup complete.')
print('GPU detected:', gpu_build)
print('KataGo binary:', cpp / 'katago')
print('Model bytes:', model.stat().st_size)


In [ ]:
import getpass, secrets, shutil, re, urllib.request

token = getpass.getpass('Enter a long random API token (at least 32 characters): ').strip()
if len(token) < 32:
    raise ValueError('Token must be at least 32 characters.')

cloud = WORK / 'cloud'
katago_bin = pathlib.Path('/kaggle/working/KataGo/cpp/katago')
config = pathlib.Path('/kaggle/working/KataGo/cpp/configs/analysis_example.cfg')
model = cloud / 'models' / 'model.bin.gz'
env = os.environ.copy()
env.update({
    'KATAGO_BIN': str(katago_bin),
    'KATAGO_CONFIG': str(config),
    'KATAGO_MODEL': str(model),
    'KATAGO_API_TOKEN': token,
    'KATAGO_MAX_CONCURRENT': '1',
    'KATAGO_DEFAULT_MAX_VISITS': '1200',
    'KATAGO_HARD_MAX_VISITS': '100000',
    'HOST': '127.0.0.1',
    'PORT': '8080',
})
run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=cloud)

log_dir = pathlib.Path('/kaggle/working/katago-cloud-logs')
log_dir.mkdir(exist_ok=True)
server_log = open(log_dir / 'uvicorn.log', 'w')
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'katago_cloud.main:app', '--host', '127.0.0.1', '--port', '8080', '--workers', '1'], cwd=cloud, env=env, stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)

import urllib.request
ready = False
for _ in range(90):
    if server.poll() is not None:
        print((log_dir / 'uvicorn.log').read_text()[-5000:])
        raise RuntimeError('KataGo Cloud API failed to start.')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/v1/healthz', timeout=2) as response:
            ready = response.status == 200
            if ready: break
    except Exception:
        time.sleep(2)
if not ready:
    raise TimeoutError('KataGo Cloud API did not become ready in time.')

cloudflared = pathlib.Path('/kaggle/working/cloudflared')
if not cloudflared.exists():
    run(['curl', '-fL', '--retry', '3', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-o', str(cloudflared)])
    cloudflared.chmod(0o755)
tunnel_log = open(log_dir / 'cloudflared.log', 'w')
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8080'], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)

public_url = None
for _ in range(90):
    if tunnel.poll() is not None:
        print((log_dir / 'cloudflared.log').read_text()[-5000:])
        raise RuntimeError('Cloudflare tunnel failed to start.')
    log_text = (log_dir / 'cloudflared.log').read_text(errors='ignore')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        public_url = match.group(0)
        break
    time.sleep(2)
if not public_url:
    raise TimeoutError('Could not obtain a temporary HTTPS tunnel URL.')

print('\nKATAGO CLOUD IS ONLINE')
print('Cloud URL:', public_url)
print('Paste this URL into KataGo Master → Engine → Cloud server URL.')
print('Paste the API token you entered above into the Cloud API Token field.')
print('Backend:', 'CUDA GPU' if gpu_build else 'CPU Eigen (no CUDA compiler/GPU detected)')
print('Keep this Kaggle session running. The URL changes after the session restarts.')
print('To stop: interrupt the notebook session.')


## Notes
- This is an on-demand endpoint, not a permanent server. The public tunnel URL is temporary and changes when the session restarts.
- The API is protected by the token you entered. Keep it private and stop the session when finished.
- A Kaggle GPU session may be unavailable, throttled or interrupted. If CUDA compilation fails, inspect the build output; do not assume GPU mode is active just because a GPU was selected in notebook settings.
- The bundled cloud service serializes analyses through one KataGo process. It is intentionally conservative for stability.
- Use identical rules, komi, model and visit budgets when comparing playing strength.
